[English](https://colab.research.google.com/github/kanoyo-git/NVC/blob/main/NVC.ipynb) | [Русский](https://colab.research.google.com/github/kanoyo-git/NVC/blob/main/NVC.ru.ipynb)

# NVC on Google Colab

NVC is a local RVC-based toolkit for voice conversion and model training. It includes single-file and batch inference, vocal separation, and dataset-aware dynamic autotune.

This notebook creates an isolated Python 3.12 environment, downloads the required v2 assets and launches NVC Studio.

Select a GPU runtime and run cells 0–6 in order. Remote access uses LocalTunnel by default.


In [ ]:
# @title 0 · GPU check
import os, shutil, subprocess, sys
assert shutil.which("nvidia-smi"), "Runtime -> Change runtime type -> GPU"
subprocess.run(["nvidia-smi"], check=True)
print("python:", sys.version.split()[0])
print("cuda visible:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))


In [ ]:
# @title 1 · Settings
from pathlib import Path

NVC_GIT_URL = "https://github.com/kanoyo-git/NVC.git"  # @param {type:"string"}
NVC_GIT_REF = "main"  # @param {type:"string"}
PROJECT_DIR = "/content/NVC"  # @param {type:"string"}
VENV_DIR = "/content/nvc-venv"  # @param {type:"string"}
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/NVC-cache"  # @param {type:"string"}
INTERFACE = "studio"  # @param ["studio", "legacy"]
ACCESS = "localtunnel"  # @param ["colab_iframe", "colab_window", "self_hosted", "cloudflared", "localtunnel", "bore"]
GUI_PORT = 7865  # @param {type:"integer"}

PROJECT = Path(PROJECT_DIR)
VENV = Path(VENV_DIR)
PY = VENV / "bin" / "python"
UV_BIN = None
print("interface:", INTERFACE)
print("access:", ACCESS)


In [ ]:
# @title 2 · System packages and uv
import os, shutil, subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

def resolve_uv():
    extra = [Path.home()/".local"/"bin", Path("/root/.local/bin"), Path.home()/".cargo"/"bin", Path("/usr/local/bin")]
    os.environ["PATH"] = os.pathsep.join(str(p) for p in extra if p.is_dir()) + os.pathsep + os.environ["PATH"]
    found = shutil.which("uv")
    if found:
        return Path(found)
    for folder in extra:
        cand = folder / "uv"
        if cand.is_file():
            return cand
    return None

sh(["apt-get", "update", "-qq"])
sh(["apt-get", "install", "-y", "-qq", "ffmpeg", "libsndfile1", "libportaudio2", "git", "unzip", "curl"])
UV_BIN = resolve_uv()
if UV_BIN is None:
    install_dir = Path.home()/".local"/"bin"
    install_dir.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy()
    env["UV_INSTALL_DIR"] = str(install_dir)
    env["UV_NO_MODIFY_PATH"] = "1"
    sh("curl -LsSf https://astral.sh/uv/install.sh | sh", shell=True, env=env)
    UV_BIN = resolve_uv()
if UV_BIN is None:
    sh(["python3", "-m", "pip", "install", "-q", "uv"])
    UV_BIN = resolve_uv()
if UV_BIN is None:
    raise FileNotFoundError("uv binary not found")
os.environ["PATH"] = str(UV_BIN.parent) + os.pathsep + os.environ["PATH"]
sh([str(UV_BIN), "--version"])
print("uv:", UV_BIN)


In [ ]:
# @title 3 · Fetch / update project
import os, subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

if USE_DRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    Path(DRIVE_CACHE_DIR).mkdir(parents=True, exist_ok=True)

if (PROJECT / ".git").is_dir():
    sh(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", NVC_GIT_REF])
    sh(["git", "-C", str(PROJECT), "checkout", "-f", "FETCH_HEAD"])
    print("updated", PROJECT)
elif (PROJECT / "gui.py").is_file():
    print("using existing project at", PROJECT)
elif NVC_GIT_URL.strip():
    if PROJECT.exists():
        sh(["rm", "-rf", str(PROJECT)])
    sh(["git", "clone", "--depth", "1", "--branch", NVC_GIT_REF, NVC_GIT_URL.strip(), str(PROJECT)])
else:
    raise SystemExit("Set NVC_GIT_URL or place gui.py in PROJECT_DIR")

os.chdir(PROJECT)
print("cwd:", Path.cwd())


In [ ]:
# @title 4 · Python environment
import os, subprocess
from pathlib import Path

def sh(cmd, **kwargs):
    print("+", cmd if isinstance(cmd, str) else " ".join(map(str, cmd)))
    subprocess.run(cmd, check=True, **kwargs)

os.chdir(PROJECT)
req_src = PROJECT / "requirments_cu128_py312.txt"
req_clean = Path("/tmp/nvc-cu128-pypi.txt")
lines = [ln for ln in req_src.read_text(encoding="utf-8").splitlines() if not ln.strip().startswith(("--index-url", "--extra-index-url"))]
req_clean.write_text("\n".join(lines) + "\n", encoding="utf-8")
if not PY.is_file():
    sh([str(UV_BIN), "venv", str(VENV), "--python", "3.12", "--clear"])
sh([str(UV_BIN), "pip", "install", "--python", str(PY), "torch==2.7.1+cu128", "torchaudio==2.7.1+cu128", "--index-url", "https://download.pytorch.org/whl/cu128"])
sh([str(UV_BIN), "pip", "install", "--python", str(PY), "-r", str(req_clean), "huggingface_hub>=0.25,<1", "--index-url", "https://pypi.org/simple"])
sh([str(UV_BIN), "pip", "install", "--python", str(PY), "noisereduce>=2.0.0", "soxr>=0.3.0", "--index-url", "https://pypi.org/simple"])
sh([str(PY), "-c", "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'ok', torch.cuda.is_available())"])


In [ ]:
# @title 5 · Download v2 assets
import codecs, os, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

os.chdir(PROJECT)
if USE_DRIVE_CACHE:
    hf_home = Path(DRIVE_CACHE_DIR) / "huggingface"
    hf_home.mkdir(parents=True, exist_ok=True)
    os.environ["HF_HOME"] = str(hf_home)

MODEL_REPO = codecs.decode("yw1995/IbvprPbairefvbaJroHV", "rot_13")
for rel in ("assets/hubert_base", "assets/rmvpe", "assets/pretrained_v2", "assets/pymss_weights", "assets/weights", "assets/indices", "logs", ".model-downloads"):
    (PROJECT / rel).mkdir(parents=True, exist_ok=True)

snapshot_download(MODEL_REPO, revision="main", allow_patterns=["hubert_base/*"], local_dir=str(PROJECT / "assets"))
hf_hub_download(MODEL_REPO, "rmvpe.pt", revision="main", local_dir=str(PROJECT / "assets" / "rmvpe"))
snapshot_download(MODEL_REPO, revision="main", allow_patterns=["pretrained_v2/*"], local_dir=str(PROJECT / "assets"))
mute_zip = hf_hub_download(MODEL_REPO, "mute.zip", revision="main", local_dir=str(PROJECT / ".model-downloads"))
snapshot_download(MODEL_REPO, revision="main", allow_patterns=["pymss_weights/*"], local_dir=str(PROJECT / "assets"))
with zipfile.ZipFile(mute_zip) as archive:
    archive.extractall(PROJECT / "logs")

# Pre-download alternative embedders (ContentVec / SPIN / SPIN-v2) so the first
# inference/training run does not depend on the network. hubert_base ships above.
# Weights are mirrored from IAHispano/Applio (MIT-licensed); NVC downloads them
# on first use via infer/hubert.py — this just warms the cache.
import urllib.request
EMBEDDER_BASE = "https://huggingface.co/IAHispano/Applio/resolve/main/Resources/embedders"
for embedder in ("contentvec", "spin", "spin-v2"):
    embedder_dir = PROJECT / "assets" / embedder
    embedder_dir.mkdir(parents=True, exist_ok=True)
    for file_name in ("pytorch_model.bin", "config.json"):
        target = embedder_dir / file_name
        if target.is_file():
            continue
        url = f"{EMBEDDER_BASE}/{embedder}/{file_name}"
        print(f"downloading {embedder}/{file_name} ...")
        with urllib.request.urlopen(url) as response, open(target, "wb") as out:
            while True:
                chunk = response.read(1 << 20)
                if not chunk:
                    break
                out.write(chunk)

required = [
    PROJECT/"assets/hubert_base/config.json",
    PROJECT/"assets/hubert_base/preprocessor_config.json",
    PROJECT/"assets/hubert_base/pytorch_model.bin",
    PROJECT/"assets/rmvpe/rmvpe.pt",
    PROJECT/"assets/pretrained_v2/f0G40k.pth",
    PROJECT/"assets/pretrained_v2/f0D40k.pth",
    PROJECT/"assets/contentvec/pytorch_model.bin",
    PROJECT/"assets/contentvec/config.json",
    PROJECT/"logs/mute",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("missing assets:\n" + "\n".join(missing))
# Optional 48k pretrains for higher-quality training; 40k is always available.
pretrained_48k = [
    PROJECT/"assets/pretrained_v2/f0G48k.pth",
    PROJECT/"assets/pretrained_v2/f0D48k.pth",
]
missing_48k = [str(p) for p in pretrained_48k if not p.exists()]
if missing_48k:
    print("note: 48k pretrains not found, 48k training will download them on demand:", missing_48k)
else:
    print("48k pretrains ready")
print("v2 assets ready")


In [ ]:
# @title 6 · Launch
import json, os, re, shutil, socket, subprocess, sys, threading, time, urllib.request
from pathlib import Path
from google.colab import output

os.chdir(PROJECT)

def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(0.4)
        return sock.connect_ex(("127.0.0.1", port)) == 0

def pump(stream):
    for line in stream:
        sys.stdout.write(line)
        sys.stdout.flush()

def wait_port(port, proc=None, timeout=180):
    deadline = time.time() + timeout
    while time.time() < deadline:
        if proc is not None and proc.poll() is not None:
            raise RuntimeError("process exited with status %s" % proc.returncode)
        if port_open(port):
            return
        time.sleep(0.4)
    raise TimeoutError("port %s did not open" % port)

def start_app():
    env = os.environ.copy()
    env["NVC_OFFLINE_CUDA_GRAPH"] = "0"
    env["PYTHONUNBUFFERED"] = "1"
    env["MPLBACKEND"] = "Agg"
    cmd = [str(PY), "-u", "gui.py", "--colab", "--noautoopen", "--port", str(GUI_PORT)]
    if INTERFACE == "legacy":
        cmd.append("--legacy")
    print("launch:", " ".join(cmd))
    proc = subprocess.Popen(cmd, cwd=str(PROJECT), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
    wait_port(GUI_PORT, proc)
    return proc

def ensure_cloudflared():
    bin_path = Path("/tmp/cloudflared")
    if not bin_path.is_file():
        subprocess.run(["curl", "-L", "--fail", "-o", str(bin_path), "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        bin_path.chmod(0o755)
    return bin_path

def ensure_bore():
    bin_path = Path("/tmp/bore")
    if not bin_path.is_file():
        archive = Path("/tmp/bore.tar.gz")
        subprocess.run(["curl", "-L", "--fail", "-o", str(archive), "https://github.com/ekzhang/bore/releases/download/v0.6.0/bore-v0.6.0-x86_64-unknown-linux-musl.tar.gz"], check=True)
        subprocess.run(["tar", "-xzf", str(archive), "-C", "/tmp"], check=True)
        bin_path.chmod(0o755)
    return bin_path

def ensure_frpc():
    version = "0.68.0"
    bin_path = Path("/tmp/frpc")
    if not bin_path.is_file():
        archive = Path("/tmp/frp.tar.gz")
        folder = Path("/tmp") / f"frp_{version}_linux_amd64"
        url = f"https://github.com/fatedier/frp/releases/download/v{version}/frp_{version}_linux_amd64.tar.gz"
        subprocess.run(["curl", "-L", "--fail", "--retry", "3", "-o", str(archive), url], check=True)
        subprocess.run(["tar", "-xzf", str(archive), "-C", "/tmp"], check=True)
        shutil.copy2(folder / "frpc", bin_path)
        bin_path.chmod(0o755)
    return bin_path

def first_url(proc, pattern, timeout=90):
    compiled = re.compile(pattern)
    deadline = time.time() + timeout
    while time.time() < deadline:
        line = proc.stdout.readline()
        if not line:
            if proc.poll() is not None:
                break
            continue
        sys.stdout.write(line)
        match = compiled.search(line)
        if match:
            threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
            return match
    threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
    raise RuntimeError("tunnel did not print a public URL")

def start_cloudflared(port):
    proc = subprocess.Popen([str(ensure_cloudflared()), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:%s" % port], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    match = first_url(proc, r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com")
    return match.group(0)

def start_localtunnel(port):
    if not shutil.which("npx"):
        raise RuntimeError("npx is not available in this runtime")
    proc = subprocess.Popen(["npx", "--yes", "localtunnel", "--port", str(port)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    match = first_url(proc, r"https://\S+")
    return match.group(0)

def start_bore(port):
    proc = subprocess.Popen([str(ensure_bore()), "local", str(port), "--to", "bore.pub"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    match = first_url(proc, r"bore\.pub:(\d+)")
    return "http://bore.pub:%s" % match.group(1)

def start_self_hosted(port):
    request = urllib.request.Request(
        "https://bootstrap.nvc.kanoyo.qzz.io:28443/tunnel/enroll",
        data=b"{}",
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=30) as response:
        enrollment = json.load(response)
    if not enrollment.get("ok"):
        raise RuntimeError(enrollment.get("error") or "tunnel enrollment failed")

    def toml(value):
        return json.dumps(str(value))

    config_path = Path("/tmp/nvc-frpc.toml")
    config_path.write_text(
        "\n".join([
            f"serverAddr = {toml(enrollment['server'])}",
            f"serverPort = {int(enrollment['server_port'])}",
            f"user = {toml(enrollment['session'])}",
            f"metadatas.ticket = {toml(enrollment['ticket'])}",
            "loginFailExit = true",
            "transport.tls.enable = true",
            "transport.tls.serverName = \"bootstrap.nvc.kanoyo.qzz.io\"",
            "transport.tls.trustedCaFile = \"/etc/ssl/certs/ca-certificates.crt\"",
            "",
            "[[proxies]]",
            "name = \"studio\"",
            "type = \"http\"",
            "localIP = \"127.0.0.1\"",
            f"localPort = {int(port)}",
            f"subdomain = {toml(enrollment['session'])}",
            "transport.useEncryption = true",
            "healthCheck.type = \"tcp\"",
            "healthCheck.intervalSeconds = 10",
            "healthCheck.timeoutSeconds = 3",
            "",
        ]),
        encoding="utf-8",
    )
    config_path.chmod(0o600)
    proc = subprocess.Popen(
        [str(ensure_frpc()), "-c", str(config_path)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    threading.Thread(target=pump, args=(proc.stdout,), daemon=True).start()
    time.sleep(3)
    if proc.poll() is not None:
        raise RuntimeError("self-hosted tunnel exited with status %s" % proc.returncode)
    return enrollment["public_url"]

if not port_open(GUI_PORT):
    start_app()
else:
    print("reusing process on", GUI_PORT)

if ACCESS == "colab_iframe":
    print("Colab iframe. Live logs work here.")
    output.serve_kernel_port_as_iframe(GUI_PORT, height=920)
elif ACCESS == "colab_window":
    print("Colab window.")
    output.serve_kernel_port_as_window(GUI_PORT)
elif ACCESS == "self_hosted":
    print("Self-hosted NVC tunnel. Each runtime gets a private random URL.")
    print(start_self_hosted(GUI_PORT))
elif ACCESS == "cloudflared":
    print("Cloudflare Quick Tunnel: no account, but SSE live logs are unsupported.")
    print(start_cloudflared(GUI_PORT))
elif ACCESS == "localtunnel":
    print("LocalTunnel: no account.")
    print(start_localtunnel(GUI_PORT))
elif ACCESS == "bore":
    print("Bore: no account.")
    print(start_bore(GUI_PORT))
else:
    raise ValueError(ACCESS)
